# Debugger AI Agent Architecture for SQL AI Systems

In this notebook, we will simulate the functionality of a debugger AI agent for debugging runtime execution error.

<figure>
 <img src="../assets/chapter_4_02.png" width="70%" align="center"/></a>
<figcaption>  Debugger AI Agent Architecture </figcaption>
</figure>

<br>
<br />


## Settings

Connecting to DuckDB:

In [7]:
import sys
import os

tbl_name = "air_traffic"

current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from sql_ai_agent.data import get_ibis_connection

csv_path = project_root + "/data/air_traffic_gold.csv"
con = get_ibis_connection(
    backend="duckdb",
    duckdb_csv_path=csv_path,
)


LLM settings:

In [8]:
from langchain_openai import ChatOpenAI

base_url = "https://api.openai.com/v1"
api_key = os.getenv("OPENAI_API_KEY")
model = "gpt-4o"

llm = ChatOpenAI(base_url=base_url, api_key=api_key, temperature=0, model=model)


We will simulate the question and the expected query:

In [9]:
question = "Show me all rows where the terminal field equals Terminal 1."

In [10]:
query = """
SELECT * 
  FROM air_traffic
WHERE
  'Terminal' = "Terminal 1"
LIMIT 5
"""

In [11]:
def test_query(con, query):
    error_message = None

    try:
        print(con.sql(query).execute())
    except Exception as e:
        print("Could not execute the query")
        error_message = e
        print(e)
    return error_message



In [12]:
error_message = test_query(con, query)

Could not execute the query
Binder Error: Referenced column "Terminal 1" not found in FROM clause!
Candidate bindings: "Terminal", "Operating Airline", "Year", "Operating Airline IATA Code", "Passenger Count"

LINE 5:   'Terminal' = "Terminal 1"
                       ^


## Setting a Prompt Template

The debugger agent prompt template settings:

In [13]:
system_template = """
You are a senior data engineer debugging SQL queries.

Your task:
- Fix the SQL query so it executes successfully.
- Do NOT repeat mistakes from previous attempts.
- Use the debug history to identify patterns or incorrect assumptions.

Rules:
- Return ONLY the corrected SQL query.
- No explanations, no markdown.

Table:
CREATE TABLE {tbl_name} ({schema})
"""

In [14]:
user_template = """
User question:
{question}

Previous debug attempts:
{debug_memory}

Latest failed query:
{query}

Error message:
{error}

Based on the history above, fix the query.
"""

In [15]:
from langchain_core.prompts import ChatPromptTemplate

messages = [("system", system_template), ("user", user_template)]

prompt = ChatPromptTemplate.from_messages(messages)


In [16]:
chain = prompt | llm


In [17]:
from sql_ai_agent.db_handler import get_tbl_attr

tbl_attr = get_tbl_attr(con=con, tbl_name=tbl_name)

schema = tbl_attr.schema

debug_memory = ""

In [18]:
llm_response = chain.invoke({
  "tbl_name": tbl_name,
  "schema": schema,
  "question": question,
  "query": query,
  "error": error_message, 
  "debug_memory": debug_memory
})

In [19]:
fixed_query = llm_response.content
print(fixed_query)

SELECT * 
FROM air_traffic
WHERE Terminal = 'Terminal 1'
LIMIT 5


Let's now test the fixed query:

In [20]:
error_message = test_query(con, fixed_query)


   Unnamed: 0  Year        Date Operating Airline Operating Airline IATA Code  \
0           0  1999  1999-07-01      ATA Airlines                          TZ   
1           1  1999  1999-07-01      ATA Airlines                          TZ   
2           2  1999  1999-07-01      ATA Airlines                          TZ   
3           5  1999  1999-07-01        Air Canada                          AC   
4           6  1999  1999-07-01        Air Canada                          AC   

  Published Airline Published Airline IATA Code    GEO Summary GEO Region  \
0      ATA Airlines                          TZ       Domestic         US   
1      ATA Airlines                          TZ       Domestic         US   
2      ATA Airlines                          TZ       Domestic         US   
3        Air Canada                          AC  International     Canada   
4        Air Canada                          AC  International     Canada   

  Activity Type Code Price Category Code    Termin